# EpistasisBench metrics

Two tasks are evaluated per assay and then averaged over assays:

1. **Task 1: Zero-Shot Epistatic Mutation Effect Prediction** — Spearman ρ between model scores and measured `DMS_score` (ΔΔG) of double mutants, restricted to the `sign` / `significant` epistasis subsets (|thermodynamic coupling| > 0.5), plus the same correlation on their single-mutant components.
2. **Task 2: Zero-Shot ΔΔG_int Scoring** — the model's non-additive log-probability score `S_int` is reconstructed from single-mutant scores,
   `S_int = score(AB) - score(A) - score(B)`, and compared with the measured `ddG_int`.
   Reported metrics: Spearman ρ, partial ρ (conditioning on `ddG_A + ddG_B`), precision, recall, F1,
   MCC, AUC, AUPRC, and top 1 / 2.5 / 5 / 10 % recall.

For each assay the metrics are computed over all mutants of that assay; the final table reports the mean
over assays (and the standard deviation of the correlation metrics).

## Input 1: label table

One row per double mutant. Default: `data/EpistasisBench.csv` produced by
`data/dataset_construction/dataset_construction.ipynb`.

| column | type | description |
|---|---|---|
| `uniprot_id` | str | UniProt accession, e.g. `AMFR_HUMAN` |
| `pdb_id` | str | 4-character PDB id, e.g. `4G3O` |
| `double_mutant` | str | `A123B:C456D`, 1-based positions, `:` separated |
| `mutation_A`, `mutation_B` | str | the two single mutants, e.g. `A123B` |
| `ddG_AB`, `ddG_A`, `ddG_B` | float | measured ΔΔG of the double and of each single mutant |
| `ddG_int` | float | `ddG_AB - ddG_A - ddG_B` |
| `thermodynamic_coupling_value` | float | experimental coupling; `|value| > 0.5` defines the `significant` subset |
| `is_sign_epistasis` | bool | sign-epistasis flag (ε = 0.07) |
| `is_significant_epistasis` | bool | `|thermodynamic_coupling_value| > 0.5` |

The notebook derives the assay id from `uniprot_id` + `pdb_id`:
`dms = f'{uniprot_id}_Tsuboyama_2023_{pdb_id}'`.

## Input 2: score tables

One csv per assay, named `{dms}.csv`, all of them in `SCORE_DIR`.

| column | type | description |
|---|---|---|
| `mutant` | str | `A123B` for a single mutant, `A123B:C456D` for a double mutant |
| `DMS_score` | float | measured ΔΔG copied from the ProteinGym substitution file (ranking target of Task 1) |
| `<model name>` | float | one column per model, must match the names in `MODEL_LIST` |

Requirements:

- every double mutant and both of its single mutants must be present in the same file, otherwise
  `S_int` cannot be reconstructed and the row is skipped (`NaN`);
- `mutant` must use 1-based positions and `:` as separator, matching the label table;
- columns may come from different scoring runs, but they have to be merged into one csv per assay
  before running this notebook.

In [ ]:
from pathlib import Path

# ------------------------------------------------------------------ paths
ROOT = Path.cwd().parents[1]                       # repository root if run from epistasisbench/evaluation/
LABEL_CSV = ROOT / 'data' / 'EpistasisBench.csv'    # see the label table description above
SCORE_DIR = ROOT / 'outputs' / 'scores'            # {dms}.csv, one file per assay
OUT_CSV = ROOT / 'outputs' / 'metrics.csv'         # aggregated metrics

# models to evaluate, must match the score columns; keep the paper naming
MODEL_LIST = ['SaProt_650M_AF2', 'ESM3', 'S3F', 'ProSST-2048']

MIN_MUTANTS = 30          # skip a subset with fewer double mutants
EPS_SIGN = 0.07           # sign epistasis margin, only used if labels are recomputed here
TC_THRESHOLD = 0.5        # |thermodynamic coupling| above this is treated as significant

In [ ]:
import numpy as np
import pandas as pd
from pandas import DataFrame, read_csv
from scipy.stats import spearmanr
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score,
)
from tqdm.auto import tqdm


def calc_top_n_percent_recall(y_true, y_pred, percent):
    """Overlap between the true top-k and the predicted top-k, k = percent% of the samples."""
    n_samples = len(y_true)
    if n_samples == 0:
        return np.nan
    k = int(n_samples * percent / 100.0)
    k = min(max(k, 1), n_samples)
    true_top_k_indices = set(np.argsort(-y_true)[:k])
    pred_top_k_indices = set(np.argsort(-y_pred)[:k])
    return len(true_top_k_indices.intersection(pred_top_k_indices)) / k


def partial_spearman_corr(x, y, z):
    """Spearman correlation between x and y after removing the effect of the confounder z."""
    r_xy, _ = spearmanr(x, y)
    r_xz, _ = spearmanr(x, z)
    r_yz, _ = spearmanr(y, z)
    return (r_xy - r_xz * r_yz) / np.sqrt((1 - r_xz ** 2) * (1 - r_yz ** 2))

In [ ]:
labels = read_csv(LABEL_CSV)
labels['dms'] = labels.uniprot_id + '_Tsuboyama_2023_' + labels.pdb_id
print(labels.shape, labels.dms.nunique(), 'assays')
labels.head(3)

## Task 1: zero-shot epistatic ΔΔG prediction

Spearman ρ on the double mutants of the `sign` and `significant` epistasis subsets, and on the
single-mutant components of the same subsets (columns with the `_single` suffix).
Assays with 30 or fewer mutations in a subset are skipped.

In [ ]:
labels['significant'] = labels.thermodynamic_coupling_value.abs() > TC_THRESHOLD

rt = []
for dms in tqdm(sorted(set(labels.dms))):
    dt = read_csv(SCORE_DIR / f'{dms}.csv')[['mutant', 'DMS_score'] + MODEL_LIST]
    df = labels.loc[labels.dms == dms]
    subsets = {'sign': df.loc[df.is_sign_epistasis], 'significant': df.loc[df.significant]}

    for name, sub in subsets.items():
        if len(sub.index) < MIN_MUTANTS:
            continue
        for model in MODEL_LIST:
            tp = dt.loc[dt.mutant.isin(sub.double_mutant)]
            corr, _ = spearmanr(tp.DMS_score, tp[model])
            rt.append((dms, model, name, corr))

            tp = dt.loc[dt.mutant.isin(sub.mutation_A) | dt.mutant.isin(sub.mutation_B)]
            corr, _ = spearmanr(tp.DMS_score, tp[model])
            rt.append((dms, model, f'{name}_single', corr))

rt_subs = DataFrame(rt, columns=['dms', 'model', 'dataset', 'spearman'])

sub_res = DataFrame()
for model in MODEL_LIST:
    for dataset in ['sign', 'significant']:
        for suffix in ['', '_single']:
            values = rt_subs.loc[(rt_subs.model == model) & (rt_subs.dataset == dataset + suffix)].spearman
            sub_res.at[model, dataset + suffix] = values.mean()
            sub_res.at[model, dataset + suffix + '_sd'] = values.std(ddof=1)

sub_res.map(lambda x: np.round(x, 3))

## Task 2: zero-shot ΔΔG_int scoring

The non-additive score is reconstructed from the model's single-mutant scores,
`S_int = score(AB) - score(A) - score(B)`, and compared with the measured `ddG_int`.
`partial_spearman` conditions on the additive term `ddG_A + ddG_B`; AUC uses stabilizing
interactions (`ddG_int > 0`) as the positive class.

In [ ]:
COLS = [
    'dms', 'model', 'spearman', 'partial_spearman', 'precision', 'recall', 'f1', 'mcc',
    'auc', 'auprc', 'top1_recall', 'top2.5_recall', 'top5_recall', 'top10_recall',
]

rt = []
for dms in tqdm(sorted(set(labels.dms))):
    dt = read_csv(SCORE_DIR / f'{dms}.csv')
    tp_all = labels.loc[labels.dms == dms].copy()

    for model in MODEL_LIST:
        mutant2score = dict(zip(dt.mutant, dt[model]))

        tp = tp_all.copy()
        tp['score'] = tp.double_mutant.map(mutant2score)
        tp['score_A'] = tp.mutation_A.map(mutant2score)
        tp['score_B'] = tp.mutation_B.map(mutant2score)

        tp['S_int'] = tp.score - tp.score_A - tp.score_B

        y_true = tp.ddG_int.values
        y_pred = tp.S_int.values

        corr, _ = spearmanr(y_true, y_pred)
        confounder = tp.ddG_A.values + tp.ddG_B.values
        partial_corr = partial_spearman_corr(y_true, y_pred, confounder)

        y_true_bin = (y_true > 0).astype(int)
        y_pred_bin = (y_pred > 0).astype(int)
        if len(np.unique(y_true_bin)) == 2:
            prec = precision_score(y_true_bin, y_pred_bin, zero_division=0)
            rec = recall_score(y_true_bin, y_pred_bin, zero_division=0)
            f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
            mcc = matthews_corrcoef(y_true_bin, y_pred_bin)
            auc = roc_auc_score(y_true_bin, y_pred)
            auprc = average_precision_score(y_true_bin, y_pred)
        else:
            prec = rec = f1 = mcc = auc = auprc = np.nan

        rt.append((
            dms, model, corr, partial_corr, prec, rec, f1, mcc, auc, auprc,
            calc_top_n_percent_recall(y_true, y_pred, 1),
            calc_top_n_percent_recall(y_true, y_pred, 2.5),
            calc_top_n_percent_recall(y_true, y_pred, 5),
            calc_top_n_percent_recall(y_true, y_pred, 10),
        ))

rt_epi = DataFrame(rt, columns=COLS)
rt_epi.head()

In [ ]:
METRICS = COLS[2:]

epi_res = DataFrame()
for model in MODEL_LIST:
    tp = rt_epi.loc[rt_epi.model == model]
    for col in METRICS:
        values = tp[col].dropna()
        epi_res.at[model, col] = values.mean() if len(values) else np.nan
        if 'spearman' in col and len(values):
            epi_res.at[model, f'{col}_sd'] = values.std(ddof=1)

epi_res = epi_res.loc[MODEL_LIST][['partial_spearman_sd', 'spearman_sd'] + METRICS]
epi_res = epi_res.map(lambda x: np.round(x, 3))

out = sub_res.join(epi_res, how='outer')
out.index.name = 'model'
out.to_csv(OUT_CSV)
out

## Output

`outputs/metrics.csv`, one row per model, columns:

- Task 1: `sign`, `sign_sd`, `sign_single`, `sign_single_sd`, `significant`, `significant_sd`,
  `significant_single`, `significant_single_sd` — mean Spearman ρ over assays (`_sd` = standard
  deviation over assays);
- Task 2: `partial_spearman_sd`, `spearman_sd` — standard deviation over assays;
- Task 2: `partial_spearman`, `spearman`, `precision`, `recall`, `f1`, `mcc`, `auc`, `auprc`,
  `top1_recall`, `top2.5_recall`, `top5_recall`, `top10_recall` — mean over assays.